In [ ]:
# -------------------------------
# Cell 1：讓使用者輸入股票代碼並對應公司名稱
# -------------------------------

ticker = input("請輸入股票代碼（例如 2330.TW）：")

# ticker -> 公司名稱 對照表，可以自行擴充
ticker_to_name = {
    "2330.TW": "台積電",

    "AAPL": "Apple Inc.",
    "MSFT": "Microsoft",
    "2317.TW": "鴻海",
    "2454.TW": "聯發科"
    # ...在這裡加入更多對映...
}

# 如果字典裡沒有，就直接用 ticker 本身當作 query
company = ticker_to_name.get(ticker, ticker)

print(f"✅ 已設定查詢關鍵字：{company}")

In [ ]:
pip install dateparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 2.3 MB/s eta 0:00:00


套件安裝完成後，請重新執行出現錯誤的程式碼區塊 (cell VQ5PDEOLBK95)。

In [ ]:
pip install pygooglenews

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 957.3 kB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=ae3277d0136c05b99f3d3c41bb4fab32627b5f5b3626df012bd628f55047a923
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k


套件安裝完成後，請重新執行出現錯誤的程式碼區塊 (cell VQ5PDEOLBK95)。

In [ ]:
pip install newspaper3k

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 3.5 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=74b6c80dea9359a50e35a2986b41ae2356d6fed7bc08bf31a7ab1aec5ada0910
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=d7d196d1a4f73625ac7826f3b54ca1c5f876853247fe6938c35d0539034a3bd1
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f160c508b2ccf00cc79
  Created wheel for jieba3k: filename=jieba3k-0.35.1-py3-none-any.whl size=7398380 sha256=d014afb1ed33c

套件安裝完成後，請重新執行出現錯誤的程式碼區塊 (cell VQ5PDEOLBK95)。

In [ ]:
pip install lxml[html_clean]

修正測試日期為2020-2026(需要重跑)

套件安裝完成後，請重新執行出現錯誤的程式碼區塊 (cell VQ5PDEOLBK95)。

In [ ]:
import base64

# 給 Python3 的 base64 補上 decodestring
if not hasattr(base64, 'decodestring'):
    base64.decodestring = base64.decodebytes

import dateparser
from datetime import datetime

_old = dateparser.parse

def _safe(date_string, *a, **kw):
    if isinstance(date_string, datetime):
        return date_string
    return _old(date_string, *a, **kw)

dateparser.parse = _safe


from time import mktime
from pygooglenews import GoogleNews
from newspaper import Article
import pandas as pd
from tqdm import tqdm
from datetime import datetime, timedelta
import dateparser


query = company


start_str = "2020-01-01"
end_str = "2025-06-30"

start_dt = datetime.fromisoformat(start_str)
end_dt = datetime.fromisoformat(end_str)


# 生成每個月的時間窗口
windows = []

cur = start_dt

while cur < end_dt:

    nxt = (cur + timedelta(days=30)).replace(day=1)

    if nxt > end_dt:
        nxt = end_dt

    windows.append((cur, nxt))

    cur = nxt + timedelta(days=1)


gn = GoogleNews(lang="zh", country="TW")

all_records = []


for dt_from, dt_to in tqdm(windows, desc="Date-windows"):

    feed = gn.search(query, from_=dt_from, to_=dt_to)

    entries = feed["entries"]

    while "next" in feed and feed["next"]:
        feed = gn.next(feed)
        entries.extend(feed["entries"])


    for entry in entries:

        # 這裡替換為 struct_time 方式
        if hasattr(entry, "published_parsed") and entry.published_parsed:

            pub = datetime.fromtimestamp(
                mktime(entry.published_parsed)
            )

        else:
            continue

        # 去除 timezone
        pub = pub.replace(tzinfo=None)

        if not (start_dt <= pub <= end_dt):
            continue


        # 下載正文
        text = ""

        try:

            art = Article(entry.link, language="zh")

            art.download()
            art.parse()

            text = art.text

            if "台積電" not in text and "台積電" not in entry.title:
                continue

        except:
            continue


        all_records.append({

            "title": entry.title,
            "publish_date": pub.strftime("%Y-%m-%d"),
            "source": getattr(entry, "source", {}).title or "",
            "link": entry.link,
            "content": text,

        })


df = pd.DataFrame(all_records)

df.drop_duplicates(subset="link", inplace=True)

df.to_csv(
    f"{query}_news_{start_str}_to_{end_str}.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"✅ 共抓到 {len(df)} 篇（去重後）")

Date-windows: 100%|██████████| 67/67 [49:11<00:00, 44.06s/it]

✅ 共抓到 5735 篇（去重後）


In [ ]:
# --- imports & API key --------------------------------------------------

import os, json, re, time
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key="")


news = pd.read_csv("台積電_news_2020-01-01_to_2025-06-30.csv")

news = news.dropna(subset=["title"])


# -------------------------------------------------
# SYSTEM MSG
# -------------------------------------------------

SYSTEM_MSG = (

"You are a financial NLP assistant. "
"Given a piece of news about a company, "
"return ONLY a valid minified JSON object with these float fields:\n"

"impact, subj_score, pos_prob, neg_prob, neu_prob, polarity, subjectivity\n"

"impact: 1 = clearly positive, -1 = clearly negative, 0 = unclear or mixed.\n"
"subj_score: subjective-ness in [0,1] where 0=fully objective.\n"
"polarity: sentiment polarity in [-1,1] like TextBlob.\n"
"subjectivity: again in [0,1] based on overall style.\n"

"The probabilities must satisfy pos+neg+neu = 1. Use 3-digit precision."

)


def build_prompt(row: pd.Series) -> str:

    return f"Title: {row['title']}\n\nReturn JSON:"

FileNotFoundError: [Errno 2] No such file or directory: '台積電_news_2020-01-01_to_2025-06-30.csv'

In [ ]:
# -------------------------------------------------
# Helper - call GPT with retries
# -------------------------------------------------

def call_gpt(prompt: str, model: str = "gpt-4o-mini", max_retry: int = 3, wait: int = 3):

    fields = [
        "impact",
        "subj_score",
        "pos_prob",
        "neg_prob",
        "neu_prob",
        "polarity",
        "subjectivity",
    ]

    default = {k: float("nan") for k in fields}


    for _ in range(max_retry):

        try:

            resp = client.chat.completions.create(

                model=model,

                messages=[

                    {"role": "system", "content": SYSTEM_MSG},

                    {"role": "user", "content": prompt},

                ],

                temperature=0.0,

            )

            text = resp.choices[0].message.content.strip()

            js_match = re.search(r"\{.*\}", text, re.S)

            if js_match:

                return json.loads(js_match.group())

        except Exception as e:

            print("Retrying:", e)

            time.sleep(wait)

    return default

In [ ]:
# -------------------------------------------------
# Main Loop
# -------------------------------------------------

records = []

for _, row in tqdm(news.iterrows(), total=len(news), desc="Scoring headlines"):

    js = call_gpt(build_prompt(row))

    records.append(js)


scores = pd.DataFrame(records)

news_aug = pd.concat(
    [news.reset_index(drop=True), scores],
    axis=1
)

NameError: name 'tqdm' is not defined

In [ ]:
# -------------------------------------------------
# Save results
# -------------------------------------------------

news_aug.to_csv(
    "mtk_news_with_sentiment_title_only.csv",
    index=False,
    encoding="utf-8-sig"
)

print("✅ Done! Saved to 2330_news_with_sentiment_title_only.csv")

抓股較資料

In [ ]:
import yfinance as yf
import pandas as pd

# -------------------------------
# Step 4：抓股價資料
# 放在 news_aug 產生之後
# -------------------------------

price_df = yf.download(
    ticker,
    start="2020-01-01",
    end="2025-07-15",   # 比新聞結束日多抓幾天，方便算隔日報酬
    auto_adjust=True,
    progress=False
)

price_df = price_df.reset_index()
price_df["Date"] = pd.to_datetime(price_df["Date"]).dt.tz_localize(None)

print(price_df.head())
print(price_df.columns)
print(f"共抓到 {len(price_df)} 筆股價資料")

新增新聞情緒指標

In [ ]:
# -------------------------------
# Step 5：建立 daily sentiment factor（升級版）
# 放在 news_aug 完成之後、Step 6 merge 之前
# -------------------------------

news_aug["publish_date"] = pd.to_datetime(news_aug["publish_date"])

# 建立逐篇新聞層級的情緒指標
news_aug["sentiment_score"] = news_aug["pos_prob"] - news_aug["neg_prob"]
news_aug["sentiment_strength"] = (news_aug["pos_prob"] - news_aug["neg_prob"]).abs()
news_aug["weighted_sentiment"] = news_aug["impact"] * news_aug["sentiment_strength"]

# 建立正負中性標記
news_aug["is_positive"] = (news_aug["sentiment_score"] > 0).astype(int)
news_aug["is_negative"] = (news_aug["sentiment_score"] < 0).astype(int)
news_aug["is_neutral"] = (news_aug["sentiment_score"] == 0).astype(int)

# 聚合成日頻情緒因子
daily_sent = (
    news_aug
    .groupby("publish_date", as_index=False)
    .agg({
        "sentiment_score": ["mean", "std"],
        "sentiment_strength": "mean",
        "weighted_sentiment": "mean",
        "impact": "mean",
        "is_positive": "mean",
        "is_negative": "mean",
        "is_neutral": "mean",
        "title": "count"
    })
)

# 攤平成單層欄位名稱
daily_sent.columns = [
    "publish_date",
    "sentiment_score",
    "sentiment_std",
    "sentiment_strength",
    "weighted_sentiment",
    "impact",
    "positive_ratio",
    "negative_ratio",
    "neutral_ratio",
    "news_count"
]

# 如果某天只有一篇新聞，std 會是 NaN，補成 0
daily_sent["sentiment_std"] = daily_sent["sentiment_std"].fillna(0)

print("✅ daily_sent 建立完成")
print(daily_sent.head())
print(daily_sent.columns)

把股價和情緒資料合併

In [ ]:
# -------------------------------
# Step 6：合併股價資料與情緒因子
# 放在 daily_sent 後面
# -------------------------------

# 扁平化 price_df 的 MultiIndex columns
# 讓所有欄位名稱都只有一層
if price_df.columns.nlevels > 1:
    price_df.columns = price_df.columns.get_level_values(0)

# 統一日期欄位名稱
price_df = price_df.rename(columns={"Date": "publish_date"})

merged = pd.merge(
    price_df,
    daily_sent,
    on="publish_date",
    how="left"
)

# 沒有新聞的日子，情緒先補 0；news_count 補 0
for col in [
    "sentiment_score",
    "sentiment_std",
    "sentiment_strength",
    "weighted_sentiment",
    "impact",
    "positive_ratio",
    "negative_ratio",
    "neutral_ratio"
]:
    merged[col] = merged[col].fillna(0)

merged["news_count"] = merged["news_count"].fillna(0)

print(merged.head())
print(merged.columns)

建立交易策略需要的欄位

In [ ]:
# -------------------------------
# Step 7：建立交易策略欄位
# 放在 merged 後面
# -------------------------------

merged = merged.sort_values("publish_date").copy()

# 當日收盤報酬
merged["ret_1d"] = merged["Close"].pct_change()

# 隔日報酬（用來當策略績效）
merged["next_ret"] = merged["Close"].shift(-1) / merged["Close"] - 1

# 簡單技術指標
merged["ma_5"] = merged["Close"].rolling(5).mean()
merged["ma_10"] = merged["Close"].rolling(10).mean()

# 你可以先做最簡單版本：
# 若當日平均情緒分數 > 0.1，且當日新聞數 >= 1，就隔日持有
merged["signal"] = (
    (merged["sentiment_score"] > 0.1) &
    (merged["news_count"] >= 1)
).astype(int)

print(merged[["publish_date", "Close", "sentiment_score", "news_count", "signal"]].head(15))

第六步：做最簡單回測

In [ ]:
# -------------------------------
# Step 7：建立交易策略欄位
# 放在 merged 後面
# -------------------------------

merged = merged.sort_values("publish_date").copy()

# 當日收盤報酬
merged["ret_1d"] = merged["Close"].pct_change()

# 隔日報酬（用來當策略績效）
merged["next_ret"] = merged["Close"].shift(-1) / merged["Close"] - 1

# 簡單技術指標
merged["ma_5"] = merged["Close"].rolling(5).mean()
merged["ma_10"] = merged["Close"].rolling(10).mean()

# 你可以先做最簡單版本：
# 若當日平均情緒分數 > 0.1，且當日新聞數 >= 1，就隔日持有
merged["signal"] = (
    (merged["sentiment_score"] > 0.1) &
    (merged["news_count"] >= 1)
).astype(int)

print(merged[["publish_date", "Close", "sentiment_score", "news_count", "signal"]].head(15))

畫圖與輸出結果

In [ ]:
import matplotlib.pyplot as plt

# -------------------------------
# Step 8：計算策略績效
# -------------------------------

# 計算買入持有策略的累積報酬
# 假設第一天的收盤價就是初始投資
merged["buy_hold_curve"] = (1 + merged["ret_1d"]).cumprod()

# 計算情緒策略的累積報酬
# 當 signal 為 1 時，持有；為 0 時，不持有 (報酬為 0)
merged["strategy_ret"] = merged["next_ret"] * merged["signal"]
# 累積報酬，假設初始投資為 1
merged["strategy_curve"] = (1 + merged["strategy_ret"]).cumprod()

# 將第一天的 buy_hold_curve 和 strategy_curve 都設為 1，代表初始資產
merged.loc[0, "buy_hold_curve"] = 1
merged.loc[0, "strategy_curve"] = 1


# -------------------------------
# Step 9：畫績效圖
# -------------------------------

plt.figure(figsize=(12, 6))
plt.plot(merged["publish_date"], merged["buy_hold_curve"], label="Buy & Hold")
plt.plot(merged["publish_date"], merged["strategy_curve"], label="Sentiment Strategy")
plt.legend()
plt.title(f"{ticker} Sentiment Strategy Backtest")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.grid(True)
plt.show()